# Product Mapping Quality: exploratory data analysis

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from pmq import config
from pmq.data import file_sha256, load_raw

df = load_raw(config.DATA_PATH)
print(f"SHA-256: {file_sha256(config.DATA_PATH)}")
print(f"Rows: {len(df):,}, columns: {df.shape[1]}")

SHA-256: 56f8f699c9e847356666c2eab3c3ab1244340f6a98ad08e39ea2199ebe993ad1
Rows: 20,000, columns: 15


## 1. Columns and missing values

In [2]:
missing = df.map(str.strip) == ""
missing_summary = pd.DataFrame(
    {
        "missing_count": missing.sum(),
        "missing_percent": (missing.mean() * 100).round(2),
    }
)
missing_summary

,missing_count,missing_percent
uniq_id,0,0.00
crawl_timestamp,0,0.00
product_url,0,0.00
product_name,0,0.00
product_category_tree,0,0.00
pid,0,0.00
retail_price,78,0.39
discounted_price,78,0.39
image,3,0.02
is_FK_Advantage_product,0,0.00


In [3]:
# Ratings use a text message instead of a number when no rating exists.
rating_columns = ["product_rating", "overall_rating"]
print((df[rating_columns] == "No rating available").sum())

numeric_columns = ["retail_price", "discounted_price"] + rating_columns
numeric = df[numeric_columns].replace(["", "No rating available"], pd.NA)
numeric = numeric.apply(pd.to_numeric)
numeric.describe().T.round(2)

product_rating    18151
overall_rating    18151
dtype: Int64


,count,mean,std,min,25%,50%,75%,max
retail_price,19922.0,2979.21,9009.64,35.0,666.0,1040.0,1999.0,571230.0
discounted_price,19922.0,1973.4,7333.59,35.0,350.0,550.0,999.0,571230.0
product_rating,1849.0,3.81,1.26,1.0,3.0,4.0,5.0,5.0
overall_rating,1849.0,3.81,1.26,1.0,3.0,4.0,5.0,5.0


## 2. Category tree

In [4]:
from pmq.data import level_2_path, select_clothing, split_category_path

category_levels = df["product_category_tree"].apply(split_category_path)
depth = category_levels.apply(len)
print("Offers by category depth:")
print(depth.value_counts().sort_index().to_string())

level_2 = df["product_category_tree"].apply(level_2_path)
print(f"\nDistinct level-2 paths: {level_2.nunique():,}")
print("Largest level-2 paths:")
print(level_2.value_counts().head(5).to_string())

Offers by category depth:
product_category_tree
1     328
2    1129
3    4419
4    4765
5    4911
6    3640
7     778
8      30

Distinct level-2 paths: 221
Largest level-2 paths:
product_category_tree
Clothing >> Women's Clothing                   3901
Clothing >> Men's Clothing                     1773
Jewellery >> Necklaces & Chains                1606
Automotive >> Accessories & Spare parts         925
Mobiles & Accessories >> Tablet Accessories     795


In [5]:
clothing = select_clothing(df)
print(f"Selected offers: {len(clothing):,}")
clothing["original_category"].value_counts()

Selected offers: 6,194


original_category
Clothing >> Women's Clothing    3901
Clothing >> Men's Clothing      1773
Clothing >> Kids' Clothing       520
Name: count, dtype: int64

## 3. Candidate inputs besides name and description

In [6]:
brand_missing = clothing["brand"].str.strip() == ""
brand_summary = brand_missing.groupby(clothing["original_category"]).agg(["size", "sum", "mean"])
brand_summary.columns = ["offers", "missing_brand", "missing_share"]
brand_summary["missing_share"] = brand_summary["missing_share"].round(3)
brand_summary

,offers,missing_brand,missing_share
original_category,,,
Clothing >> Kids' Clothing,520,176,0.338
Clothing >> Men's Clothing,1773,1231,0.694
Clothing >> Women's Clothing,3901,1635,0.419


In [7]:
import re

IDEAL_FOR = r'"key"=>"Ideal For", "value"=>"([^"]*)"'


def extract_ideal_for(raw_specifications):
    # Specifications are not valid JSON, so only this one key/value pattern is matched.
    matches = re.findall(IDEAL_FOR, raw_specifications)
    if len(matches) == 0:
        return pd.NA
    return matches[0].strip()


ideal_for = clothing["product_specifications"].apply(extract_ideal_for)
print("Share of offers without Ideal For:")
print(ideal_for.isna().groupby(clothing["original_category"]).mean().round(3).to_string())
print(f"\nDistinct Ideal For values: {ideal_for.nunique()}")
print("Most common values:")
print(ideal_for.value_counts().head(8).to_string())

Share of offers without Ideal For:


original_category
Clothing >> Kids' Clothing      0.040
Clothing >> Men's Clothing      0.048
Clothing >> Women's Clothing    0.011

Distinct Ideal For values: 17
Most common values:
product_specifications
Women's            3832
Men's              1682
Girl's              273
Boy's               167
Baby Boy's           33
Women's, Girl's      13
Baby Girl's          13
Girl's, Women's      12


## 4. Duplicates

In [8]:
text_columns = ["product_name", "description"]
duplicate_summary = pd.DataFrame(
    {
        "criterion": ["identical whole rows", "same pid", "same name and description"],
        "extra_rows": [
            clothing.duplicated().sum(),
            clothing.duplicated(subset=["pid"]).sum(),
            clothing.duplicated(subset=text_columns).sum(),
        ],
    }
)
duplicate_summary

,criterion,extra_rows
0,identical whole rows,0
1,same pid,1
2,same name and description,693


In [9]:
from pmq.data import drop_exact_text_duplicates

# Exact text pairs never have conflicting categories, otherwise this raises an error.
categories_per_text = clothing.groupby(text_columns)["original_category"].nunique()
print(f"Text pairs with more than one category: {(categories_per_text > 1).sum()}")

clean = drop_exact_text_duplicates(clothing)
cleanup = pd.DataFrame(
    {
        "before": clothing["original_category"].value_counts(),
        "after": clean["original_category"].value_counts(),
    }
)
cleanup["removed"] = cleanup["before"] - cleanup["after"]
print(f"Offers: {len(clothing):,} -> {len(clean):,}")
cleanup

Text pairs with more than one category: 0


Offers: 6,194 -> 5,501


,before,after,removed
original_category,,,
Clothing >> Women's Clothing,3901,3488,413
Clothing >> Men's Clothing,1773,1517,256
Clothing >> Kids' Clothing,520,496,24


## 5. Text completeness and split groups

In [10]:
for column in text_columns:
    blank = clean[column].str.strip() == ""
    print(f"Blank {column}: {blank.sum()}")

Blank product_name: 0
Blank description: 1


In [11]:
from pmq.data import normalize_product_name

group_id = clean["product_name"].apply(normalize_product_name)
group_sizes = group_id.value_counts()
print(f"Offers: {len(clean):,}")
print(f"Groups (normalized names): {len(group_sizes):,}")
print(f"Groups with more than one offer: {(group_sizes > 1).sum():,}")
print(f"Offers in those groups: {group_sizes[group_sizes > 1].sum():,}")
print("\nLargest groups:")
print(group_sizes.head(5).to_string())

Offers: 5,501
Groups (normalized names): 3,005
Groups with more than one offer: 815
Offers in those groups: 3,311

Largest groups:
product_name
nimya solid men's polo neck t-shirt                            41
grafion by grafion - comfort feel women's full coverage bra    40
s4s stylish women's push-up bra                                39
s4s comfortable women's full coverage bra                      32
candy house solid men's polo neck t-shirt                      30


## 6. Data quality issues

In [12]:
last_level = category_levels.apply(lambda levels: levels[-1])
same_as_name = last_level.str.lower() == df["product_name"].str.strip().str.lower()

print(f"Category path with only one level: {(depth == 1).sum():,}")
print(f"Category path cut off with '...': {last_level.str.endswith('...').sum():,}")
print(f"Last category level equal to the product name: {same_as_name.sum():,}")

Category path with only one level: 328
Category path cut off with '...': 1,732
Last category level equal to the product name: 2,050


In [13]:
# Offers whose name names only the other audience, e.g. "Women's" in Men's Clothing.
name = clean["product_name"].str.lower()
says_women = name.str.contains(r"\bwomen'?s\b")
says_men = name.str.contains(r"\bmen'?s\b")
category = clean["original_category"]

contradictions = pd.Series(
    {
        "Men's category, name says Women's": ((category == config.MEN) & says_women & ~says_men).sum(),
        "Women's category, name says Men's": ((category == config.WOMEN) & says_men & ~says_women).sum(),
        "Kids' category, name says Women's or Men's": (
            (category == config.KIDS) & (says_women != says_men)
        ).sum(),
    }
)
print(f"Offers checked: {len(clean):,}")
contradictions

Offers checked: 5,501


Men's category, name says Women's             2
Women's category, name says Men's             2
Kids' category, name says Women's or Men's    9
dtype: int64